<a href="https://colab.research.google.com/github/Sameed-Gillani/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sameed-Gillani/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

Unit of Analysis: One row represents the daily search performance of a single page (url_id) for a specific client (client_id) on a specific date.
Time Window: The historical daily performance spanning the 90 days prior to the decision point, which is used to evaluate the target state across the subsequent 30-day future window.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

Feature: clicks_90d, impressions_90d, content_age_days (Historical metrics and metadata fully available at the time of prediction).
Label / Proxy: target_future_crash_risk (A binary label: 1 if the page experiences a >20% traffic drop in the upcoming 30 days, 0 otherwise).
Context: client_id, url_id, report_date (Identifiers used to define the row's grain, not used as predictive features).
Excluded: Any daily traffic metrics occurring after the target decision date. These are strictly excluded to prevent data leakage from the future.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [8]:
import duckdb
from google.colab import userdata

# 1. Securely load your HF token and connect to DuckDB
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

# 2. Authenticate using DuckDB's Secret Manager
con.execute(f"CREATE SECRET flyrank_hf_secret (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

# 3. Fixed Path
path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

print("--- QUERY 1: THE GRAIN ---")
# Proving one row really is one URL (content) per client per date
con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) as row_count
    FROM '{path}'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").show()

print("\n--- QUERY 2: ROW COUNT & DATE SPAN ---")
con.sql(f"""
    SELECT
        COUNT(*) as total_rows,
        MIN(report_date) as first_date,
        MAX(report_date) as last_date
    FROM '{path}'
""").show()

print("\n--- QUERY 3: AVAILABILITY (IS TRUE) ---")
# Filtering to show how many rows survive when we only look at rows with active search data
con.sql(f"""
    SELECT COUNT(*) as surviving_active_rows
    FROM '{path}'
    WHERE gsc_data_available IS TRUE
""").show()

--- QUERY 1: THE GRAIN ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬─────────────────┬─────────────┬───────────┐
│ client_hash_id │ content_hash_id │ report_date │ row_count │
│    varchar     │     varchar     │    date     │   int64   │
├────────────────┴─────────────────┴─────────────┴───────────┤
│                           0 rows                           │
└────────────────────────────────────────────────────────────┘


--- QUERY 2: ROW COUNT & DATE SPAN ---
┌────────────┬────────────┬────────────┐
│ total_rows │ first_date │ last_date  │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘


--- QUERY 3: AVAILABILITY (IS TRUE) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────────┐
│ surviving_active_rows │
│         int64         │
├───────────────────────┤
│               3611061 │
└───────────────────────┘



## 4. Data limits

Data Limits: This dataset can never tell me if a sudden drop in traffic was caused by an external Google core algorithm update, nor does it capture social media or direct traffic (GSC only shows organic search). Additionally, because of unbalanced history, recently published pages will not have a full 90-day historical baseline to compare against, meaning my model's confidence will naturally be lower for newer content.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.